# 2016 — Byte-Pair Encoding: Subword Tokenization

**Papers:** Rico Sennrich, Barry Haddow & Alexandra Birch, *"Neural Machine Translation of Rare Words with Subword Units"*, ACL (2016). The algorithm comes from data compression: Philip Gage, *"A New Algorithm for Data Compression"* (1994). Byte-level BPE: Radford et al., GPT-2 (2019).

Every language model in this repository so far reads **characters**. Real LLMs read **tokens**, and almost all of them use BPE or a close relative (WordPiece in BERT, Unigram/SentencePiece in T5 and Llama 1–2).

The problem it solves:
- **Word-level** vocabularies are huge and still miss rare words, names and typos (the "unknown word" problem in 2015 machine translation).
- **Character-level** sequences have no unknown words but are long, so the model spends its context and compute on spelling.

**BPE** starts from characters (or, in GPT-2 onward, the 256 possible **bytes**) and repeatedly **merges the most frequent adjacent pair** into a new symbol. Frequent words become one token, rare words split into pieces, and nothing is ever out of vocabulary.

![Diagram: Byte-pair encoding (Sennrich et al., 2016; byte-level: GPT-2, 2019)](https://raw.githubusercontent.com/babushkai/neuralnetwork/main/assets/diagrams/34_2016_byte_pair_encoding.png)

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)
from collections import Counter

PyTorch 2.15.0 | device: mps


In [2]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")

290,321 characters, vocab 31


## Training BPE from scratch
Byte-level, GPT-2 style: split the text into "words" (a space attaches to the following word), turn each into its UTF-8 bytes, then merge.

In [3]:
import re
PRETOK = re.compile(r" ?[A-Za-z]+| ?[0-9]+| ?[^A-Za-z0-9\s]+|\s+")

def train_bpe(text, n_merges):
    words = Counter(tuple(w.encode("utf-8")) for w in PRETOK.findall(text))
    vocab = {i: bytes([i]) for i in range(256)}
    merges = {}                                                               # (a, b) -> new id, in order of creation
    for k in range(n_merges):
        pairs = Counter()
        for w, f in words.items():
            for a, b in zip(w, w[1:]): pairs[a, b] += f
        if not pairs: break
        (a, b), freq = pairs.most_common(1)[0]
        new = 256 + k; merges[a, b] = new; vocab[new] = vocab[a] + vocab[b]
        merged = Counter()
        for w, f in words.items():                                            # apply the merge everywhere
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: out.append(new); i += 2
                else: out.append(w[i]); i += 1
            merged[tuple(out)] += f
        words = merged
    return merges, vocab

def encode_word(b, merges):
    ids = list(b)
    while len(ids) > 1:                                                       # apply the earliest-learned merge first
        pair = min(zip(ids, ids[1:]), key=lambda p: merges.get(p, 1e18))
        if pair not in merges: break
        new, out, i = merges[pair], [], 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair: out.append(new); i += 2
            else: out.append(ids[i]); i += 1
        ids = out
    return ids

_caches = {}
def bpe_encode(s, merges):
    cache = _caches.setdefault(len(merges), {})                             # merge lists here are prefixes of one list
    ids = []
    for w in PRETOK.findall(s):
        if w not in cache: cache[w] = encode_word(w.encode("utf-8"), merges)
        ids += cache[w]
    return ids
def show(i):
    """A token as text if it is valid UTF-8 on its own, otherwise as its raw bytes."""
    try: return vocab[i].decode("utf-8")
    except UnicodeDecodeError: return "<" + vocab[i].hex(" ") + ">"
def bpe_decode(ids, vocab): return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

t0 = time.time(); merges, vocab = train_bpe(text[:split], 300)
print(f"300 merges learned in {time.time() - t0:.2f}s ({len(set(PRETOK.findall(text[:split])))} distinct pre-tokens: this corpus has a small vocabulary)")
print("first 12 merges:", [vocab[i].decode() for i in range(256, 268)])
print("merges 100–112: ", [vocab[i].decode() for i in range(356, 368)])

300 merges learned in 0.03s (58 distinct pre-tokens: this corpus has a small vocabulary)
first 12 merges: ['he', ' t', 'te', ' the', ' a', ' s', 'er', ' l', 'ts', 'en', 're', ' c']
merges 100–112:  [' key', ' an', ' and', ' v', ' val', ' valu', ' value', ' lon', ' long', ' con', ' conte', ' context']


The first merges are frequent letter pairs ("e", "th"-like pieces, " t"); later ones are whole words with their leading space, like " the" and " learns".
## Round trip, rare words, and anything at all

In [4]:
for s in ["The model learns the next token step by step.", "Transformerization of hyperparameters!", "naïve café 東京 🙂"]:
    ids = bpe_encode(s, merges)
    print(f"{len(s.encode()):3d} bytes → {len(ids):3d} tokens | {[show(i) for i in ids]}")
    assert bpe_decode(ids, vocab) == s                                      # lossless, even for unseen scripts

 45 bytes →  11 tokens | ['T', 'he', ' model', ' learns', ' the', ' next', ' token', ' step', ' by', ' step', '.']
 38 bytes →  31 tokens | ['T', 'r', 'a', 'n', 's', 'f', 'or', 'm', 'er', 'i', 'z', 'a', 'ti', 'on', ' ', 'o', 'f', ' h', 'y', 'p', 'er', 'p', 'a', 'r', 'a', 'm', 'e', 'te', 'r', 's', '!']
 24 bytes →  22 tokens | ['n', 'a', '<c3>', '<af>', 'v', 'e', ' ca', 'f', '<c3>', '<a9>', ' ', '<e6>', '<9d>', '<b1>', '<e4>', '<ba>', '<ac>', ' ', '<f0>', '<9f>', '<99>', '<82>']


Unseen words split into known pieces; characters never seen in training fall back to raw bytes (shown as `<hex>`; "ï" is two bytes, "東" three, the emoji four). There is no "unknown" token. Our corpus has only a few dozen distinct words, so after 300 merges every one of them is a single token; real tokenizers train on terabytes of diverse text and keep subword pieces for the long tail.
## Vocabulary size vs. sequence length

In [5]:
val_text = text[split:]
for n in [0, 25, 50, 100, 200, 300]:
    m = {k: v for k, v in merges.items() if v < 256 + n}
    print(f"{256 + n:4d} tokens in vocab → {len(val_text) / len(bpe_encode(val_text, m)):.2f} characters per token on held-out text")

 256 tokens in vocab → 1.00 characters per token on held-out text
 281 tokens in vocab → 1.42 characters per token on held-out text
 306 tokens in vocab → 1.71 characters per token on held-out text
 356 tokens in vocab → 2.23 characters per token on held-out text
 456 tokens in vocab → 4.99 characters per token on held-out text
 556 tokens in vocab → 5.56 characters per token on held-out text


## Does it help a language model?
Train the same tiny GPT twice, on characters (notebook 41's setup) and on BPE tokens (556 = 256 bytes + 300 merges), with the same context of 64 *tokens* and the same number of steps. Losses are converted to **bits per character** so they are comparable.

In [6]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj, self.h = nn.Linear(d, 3 * d), nn.Linear(d, d), heads
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    """A small GPT-2-style character model (pre-norm blocks, learned positions)."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return self.head(self.ln(x))

def train_gpt(model, train_data, steps=800, B=32, lr=2e-3):
    model = model.to(train_data.device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - model.ctx - 1, (B,), device=train_data.device)
        x = train_data[ix[:, None] + torch.arange(model.ctx, device=train_data.device)]; y = train_data[ix[:, None] + torch.arange(1, model.ctx + 1, device=train_data.device)]
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def eval_loss(model, data, n=20, B=32):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    for _ in range(n):
        ix = torch.randint(0, len(data) - model.ctx - 1, (B,), generator=g).to(data.device)
        x = data[ix[:, None] + torch.arange(model.ctx, device=data.device)]; y = data[ix[:, None] + torch.arange(1, model.ctx + 1, device=data.device)]
        losses.append(F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1)).item())
    return float(np.mean(losses))

In [7]:
tok = torch.tensor(bpe_encode(text, merges), device=device)
tsplit = len(bpe_encode(text[:split], merges))
tok_train, tok_val = tok[:tsplit], tok[tsplit:]
chars_per_tok = split / tsplit
results = {}
for name, d_train, d_val, vocab_size, cpt in [("characters", train_data, val_data, V, 1.0), ("BPE (556)", tok_train, tok_val, 556, chars_per_tok)]:
    seed_everything(0); t0 = time.time()
    m = train_gpt(GPT(vocab_size, ctx=64), d_train, steps=800)
    loss = eval_loss(m, d_val)
    results[name] = m
    print(f"{name:11s} val loss {loss:.3f} nats/token = {loss / math.log(2) / cpt:.3f} bits/char | context = {64 * cpt:.0f} characters | {time.time() - t0:.0f}s")

characters  val loss 0.240 nats/token = 0.346 bits/char | context = 64 characters | 22s


BPE (556)   val loss 1.037 nats/token = 0.269 bits/char | context = 356 characters | 18s


In [8]:
seed_everything(1)
idx = torch.tensor([bpe_encode("The model", merges)], device=device)
with torch.no_grad():
    for _ in range(40):
        p = F.softmax(results["BPE (556)"](idx[:, -64:])[:, -1] / 0.8, -1); idx = torch.cat([idx, torch.multinomial(p, 1)], 1)
print(bpe_decode(idx[0].tolist(), vocab))

The model predicts the loss. Each layer updates its weights in parallel, then the model predicts the next token. The optimizer compresses its weights quickly. A student predicts the next token carefully. The router learns a hidden


## Takeaways
- **BPE = greedy compression.** Merge the most frequent pair, repeat. The merge list *is* the tokenizer; encoding replays merges in order.
- **Byte-level** BPE (GPT-2 onward) can encode any string losslessly, with no unknown token.
- Each token carries several characters, so the same context window covers several times more text and each prediction step covers more of it; per character, the model is also cheaper to run.
- **Costs:** tokenization is why LLMs struggle to count letters in a word and why some languages need many more tokens per sentence than English. Modern tokenizers use 100K–260K tokens (e.g. o200k in GPT-4o, 256K in Gemma) trained on multilingual data, and research on tokenizer-free byte models (e.g. BLT, 2024) continues.